## WP011 — Multi-League Hierarchical Pooling: Validation

See `README.md` for the architecture, what was implemented, and unit-test verification. This notebook is the actual validation run — same yardstick as every WP since WP003 (pooled RPS and paired-bootstrap gap to Pinnacle closing odds on the 401-match EPL comparison), with `EPL` as `eval_league` throughout; the other leagues are training-only context.

**Compute discipline baked in, not bolted on afterward** (see the "easy wins" discussion this WP was built from):
1. **Concurrency** — windows run several-at-once via `run_windows_concurrent` (`scripts/run_cv_window.py`), not the sequential one-at-a-time loop every prior WP used. Verified race-free with 8 dedicated unit tests (`tests/test_run_cv_window.py`) before being used here.
2. **Trimming** — non-EPL leagues fetch fewer seasons than EPL's full 6. The statistical benefit of pooling is mainly "more independent teams," not "more seasons per team," so this keeps most of the benefit for a fraction of the compute.
3. **Screening before confirming** — 18 windows first (this project's standard pattern since WP005/WP006/WP009), full 35 only if screening shows something worth confirming.

**Heavy compute — the screening/confirmation cells are yours to run.** A small real correctness check (3 windows, 2 extra leagues, 2 trimmed seasons) was already run and passed before this notebook was written — see README's Verification section.

In [2]:
import json
import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

from football_model.data.get_data import get_understat_data
from football_model.features.add_metadata import add_rounds_to_data, add_match_ids, add_home_away_goals_xg
from football_model.model.predict import dc_outcome_probs
from football_model.types.model_data import ModelConfig

REPO = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP001 = REPO / 'work_products' / 'wp001_walkforward_cv_baseline'
WP003 = REPO / 'work_products' / 'wp003_bookmaker_benchmark'
WP009 = REPO / 'work_products' / 'wp009_lineup_xg_validation'
WP011 = REPO / 'work_products' / 'wp011_multileague_hierarchy'
SCRIPT = REPO / 'scripts' / 'run_cv_window.py'
sys.path.insert(0, str(REPO / 'scripts'))
from run_cv_window import run_windows_concurrent, _load_checkpoint  # noqa: E402

with open(WP001 / 'cv_shared_data.pkl', 'rb') as f:
    shared_base = pickle.load(f)
df_cv, windows = shared_base['df_cv'], shared_base['windows']
print(len(windows), 'windows;', len(df_cv), 'EPL rows; EPL seasons', sorted(df_cv['season'].unique()))

35 windows; 4560 EPL rows; EPL seasons ['2020', '2021', '2022', '2023', '2024', '2025']


### Data assembly — EPL reused as-is, other leagues fetched trimmed

`TRIM_YEARS_NON_EPL` is the actual compute lever: fewer seasons for Bundesliga/La_Liga/Serie_A/Ligue_1 than EPL's own 6 — cuts the biggest new source of compute (4 extra leagues × however many seasons) while keeping most of the "more independent teams" benefit the whole WP is betting on. Adjust the year list and OTHER_LEAGUES set based on what the timed single-window check below actually costs — don't commit to all 4 at full trim length without measuring first.

In [3]:
OTHER_LEAGUES = ['Bundesliga', 'La_Liga', 'Serie_A', 'Ligue_1']
TRIM_YEARS_NON_EPL = ['2023', '2024', '2025']  # last 3 seasons, not EPL's full 6 -- see note above

DATA_PATH = WP011 / 'cv_shared_data.pkl'

if DATA_PATH.exists():
    with open(DATA_PATH, 'rb') as f:
        shared = pickle.load(f)
    dfs_by_league = shared['dfs_by_league']
    print('loaded cached multi-league data:', {k: len(v) for k, v in dfs_by_league.items()})
else:
    dfs_by_league = {'EPL': df_cv}
    for league in OTHER_LEAGUES:
        raw = get_understat_data(years=TRIM_YEARS_NON_EPL, leagues=[league])
        df = add_home_away_goals_xg(add_match_ids(add_rounds_to_data(raw)))
        dfs_by_league[league] = df
        print(f'{league}: {len(df)} rows, seasons {sorted(df["season"].unique())}, {df["team"].nunique()} teams')

    shared = {'dfs_by_league': dfs_by_league, 'eval_league': 'EPL', 'windows': windows}
    with open(DATA_PATH, 'wb') as f:
        pickle.dump(shared, f)
    print('wrote', DATA_PATH)

loaded cached multi-league data: {'EPL': 4560, 'Bundesliga': 1836, 'La_Liga': 2280, 'Serie_A': 2280, 'Ligue_1': 1836}


### Candidate configs

Just ONE new arm for this pass — `multileague`, at WP001's own default priors (not `loose_combo`'s manually-loosened values; the point of pooling is to let the data set `sigma_att`/`sigma_def`/`home_adv`'s spread, not guess a looser number by hand). `baseline` and `lineup_loose_combo` are reused from their existing WP001/WP009 checkpoints, not re-run — same `seed_from` pattern WP009 used against WP001/WP005.

In [4]:
ARMS = {
    'baseline':           None,   # single-league, seeded from WP001 -- not run here
    'lineup_loose_combo': None,   # single-league, seeded from WP009 -- not run here
    'multileague':        {},     # NEW -- the only arm this notebook actually fits
}
BASE = dict(clip_theta=5.0, center_team_strength=False, use_dixon_coles=True, use_xG=True)
ModelConfig(**BASE)  # sanity check the base config is itself valid
print('arms:', list(ARMS))

arms: ['baseline', 'lineup_loose_combo', 'multileague']


## Phase 2 — Screening CV (18 windows)

**Heavy compute — run this yourself.** Timing note: before committing to the full screen, time ONE window first (cell below) — a real 5-league, 3-season window has not been measured end-to-end by me, only a smaller 2-league/1-season config (see README). Use that number to sanity-check `max_workers` won't exhaust RAM (each concurrent window is a full NUTS run) before launching the full 18.

In [5]:
SCREEN_WINDOWS = list(range(1, len(windows) + 1, 2))  # same convention as WP005/WP006/WP009
MAX_WORKERS = 4   # conservative starting point -- see README's compute-wins discussion
WINDOW_TIMEOUT = 1800

def load_ckpt(p):
    return pickle.load(open(p, 'rb')) if p.exists() else {'results': [], 'cv_match_predictions': []}

def seed_from(src_checkpoint_path, dest_name, windows_subset):
    dest = WP011 / f'cv_checkpoint_{dest_name}.pkl'
    if dest.exists():
        return
    src = load_ckpt(src_checkpoint_path)
    filt = {'results': [r for r in src['results'] if r['window'] in windows_subset],
            'cv_match_predictions': [m for m in src['cv_match_predictions'] if m['window'] in windows_subset]}
    pickle.dump(filt, open(dest, 'wb'))
    print(f'seeded {dest_name} from {src_checkpoint_path.name}:', len(filt['results']), 'windows')

seed_from(WP001 / 'cv_checkpoint.pkl', 'baseline', SCREEN_WINDOWS)
seed_from(WP009 / 'cv_checkpoint_lineup_loose_combo.pkl', 'lineup_loose_combo', SCREEN_WINDOWS)

In [6]:
# --- Time ONE window first (recommended before launching the full 18) ---
ckpt_multileague = WP011 / 'cv_checkpoint_multileague.pkl'
t0 = time.time()
run_windows_concurrent(
    SCRIPT, DATA_PATH, ckpt_multileague, [SCREEN_WINDOWS[0]],
    config_overrides=ARMS['multileague'], max_workers=1, timeout=WINDOW_TIMEOUT,
)
print(f'single window took {time.time()-t0:.1f}s -- use this to size max_workers/RAM before the full screen below')

  nothing to do — every requested window is already in the checkpoint
single window took 0.0s -- use this to size max_workers/RAM before the full screen below


In [7]:
# --- Full 18-window screen (concurrent) ---
t0 = time.time()
run_windows_concurrent(
    SCRIPT, DATA_PATH, ckpt_multileague, SCREEN_WINDOWS,
    config_overrides=ARMS['multileague'], max_workers=MAX_WORKERS, timeout=WINDOW_TIMEOUT,
)
print(f'\nscreening batch wall time: {(time.time()-t0)/60:.1f} min')

for name in ARMS:
    n = len(load_ckpt(WP011 / f'cv_checkpoint_{name}.pkl')['results'])
    print(f'  {name}: {n}/{len(SCREEN_WINDOWS)}')

  nothing to do — every requested window is already in the checkpoint

screening batch wall time: 0.0 min
  baseline: 18/18
  lineup_loose_combo: 18/18
  multileague: 18/18


### Phase 2 analysis — resolution + gap to Pinnacle per arm

Same helpers as WP003/WP005/WP009 — `fixtures_for` rebuilds fixture identity from a checkpoint's match predictions, `devig`/`rps_row`/`boot` are the shared RPS + paired-bootstrap machinery. Reused, not reimplemented.

In [8]:
odds_raw = pd.read_pickle(WP003 / 'odds_raw.pkl')
CODE_TO_FD = {'ARS': 'Arsenal', 'AVL': 'Aston Villa', 'BOU': 'Bournemouth', 'BRE': 'Brentford',
    'BRI': 'Brighton', 'BUR': 'Burnley', 'CHE': 'Chelsea', 'CRY': 'Crystal Palace', 'EVE': 'Everton',
    'FLH': 'Fulham', 'IPS': 'Ipswich', 'LED': 'Leeds', 'LEI': 'Leicester', 'LIV': 'Liverpool',
    'LUT': 'Luton', 'MCI': 'Man City', 'MUN': 'Man United', 'NEW': 'Newcastle', 'NOR': 'Norwich',
    'NOT': "Nott'm Forest", 'SHE': 'Sheffield United', 'SOU': 'Southampton', 'SUN': 'Sunderland',
    'TOT': 'Tottenham', 'WAT': 'Watford', 'WBA': 'West Brom', 'WHU': 'West Ham', 'WOL': 'Wolves'}
df_sorted = df_cv.sort_values('datetime').reset_index(drop=True)

def fixtures_for(ckpt, windows_list=None):
    windows_list = windows if windows_list is None else windows_list
    mp = ckpt['cv_match_predictions']
    rows = []
    for w in sorted({m['window'] for m in mp}):
        win = windows_list[w - 1]
        sel = df_sorted[(df_sorted['is_home'] == 1) & (df_sorted['round'] >= win['test_start']) & (df_sorted['round'] <= win['test_end'])]
        wp = [m for m in mp if m['window'] == w]
        for (_, r), m in zip(sel.iterrows(), wp):
            rows.append({'date': pd.Timestamp(r['datetime']).normalize(), 'home_fd': CODE_TO_FD[r['team']],
                         'away_fd': CODE_TO_FD[r['opp_team']], 'goals_home': m['goals_home'], 'goals_away': m['goals_away'],
                         'lambda_home': m['lambda_home'], 'lambda_away': m['lambda_away'], 'rho_dc': m.get('rho_dc')})
    df = pd.DataFrame(rows)
    probs = [dc_outcome_probs(r.lambda_home, r.lambda_away, rho=r.rho_dc) for r in df.itertuples()]
    df[['p_home_model', 'p_draw_model', 'p_away_model']] = np.array(probs)
    df['result'] = np.where(df['goals_home'] > df['goals_away'], 'H', np.where(df['goals_home'] == df['goals_away'], 'D', 'A'))
    return df

def devig(o):
    inv = 1.0 / np.asarray(o, float)
    return inv / inv.sum()

def rps_row(ph, pd_, pa, actual):
    cp1, cp2 = ph, ph + pd_
    ce1 = 1.0 if actual == 'H' else 0.0
    ce2 = 1.0 if actual in ('H', 'D') else 0.0
    return 0.5 * ((cp1 - ce1) ** 2 + (cp2 - ce2) ** 2)

def boot(values, n_boot=5000, seed=0):
    v = np.asarray(values, float)
    rng = np.random.default_rng(seed)
    bm = np.array([rng.choice(v, size=len(v), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(bm, [2.5, 97.5])
    return v.mean(), lo, hi

def gap_vs_pinnacle(ckpt, windows_list=None):
    df = fixtures_for(ckpt, windows_list)
    j = df.merge(odds_raw, left_on=['date', 'home_fd', 'away_fd'], right_on=['Date', 'HomeTeam', 'AwayTeam'], how='left')
    j = j.dropna(subset=['PSCH', 'PSCD', 'PSCA'])
    P = np.array([devig([r.PSCH, r.PSCD, r.PSCA]) for r in j.itertuples()])
    j['p_home_pin'], j['p_draw_pin'], j['p_away_pin'] = P[:, 0], P[:, 1], P[:, 2]
    j['rps_model'] = [rps_row(r.p_home_model, r.p_draw_model, r.p_away_model, r.result) for r in j.itertuples()]
    j['rps_pin'] = [rps_row(r.p_home_pin, r.p_draw_pin, r.p_away_pin, r.result) for r in j.itertuples()]
    gap = j['rps_model'] - j['rps_pin']
    m, lo, hi = boot(gap.values)
    return {'n': len(j), 'model_rps': j['rps_model'].mean(), 'gap': m, 'ci': (lo, hi)}

for name in ARMS:
    ckpt = load_ckpt(WP011 / f'cv_checkpoint_{name}.pkl')
    if not ckpt['results']:
        print(f'{name}: not run yet')
        continue
    r = gap_vs_pinnacle(ckpt, windows)
    print(f"{name:<20} n={r['n']:>3}  model RPS {r['model_rps']:.4f}  gap {r['gap']:+.4f}  "
          f"CI [{r['ci'][0]:+.4f}, {r['ci'][1]:+.4f}]")

baseline             n=195  model RPS 0.1916  gap +0.0129  CI [+0.0057, +0.0199]
lineup_loose_combo   n=195  model RPS 0.1903  gap +0.0117  CI [+0.0047, +0.0186]
multileague          n=195  model RPS 0.1920  gap +0.0133  CI [+0.0059, +0.0205]


### Phase 2b — full-coverage vs. partial-coverage windows

**Why this exists**: `TRIM_YEARS_NON_EPL` means non-EPL leagues only have data from a certain point onward — an early window's training cutoff can predate that, in which case `prepare_multileague_data` skips the not-yet-started league(s) for that window and it trains EPL-only (see README's "Compute" section — this was found and fixed via a real test, not assumed). That's correct behaviour, not a bug, but it means the `multileague` arm's *pooled* gap-to-Pinnacle across all windows is a blend of "no pooling benefit at all" (early windows) and "full joint-model benefit" (later windows) — diluting any real effect toward the baseline. Splitting by whether a window actually used every available league isolates the real test from the diluted one.

In [9]:
def leagues_coverage_split(ckpt):
    """Partition a multi-league checkpoint's window numbers into
    'full' (every available league was actually used -- see
    run_window_multileague's leagues_used vs leagues_available) and
    'partial' (at least one league was skipped for that window, most
    commonly because TRIM_YEARS_NON_EPL means it hadn't started yet).
    Windows with no leagues_available at all (shouldn't happen for the
    multileague arm, but guards against an unexpected checkpoint shape)
    are excluded from both."""
    full, partial = [], []
    for r in ckpt['results']:
        avail = set(r.get('leagues_available', []))
        used = set(r.get('leagues_used', []))
        if not avail:
            continue
        (full if used == avail else partial).append(r['window'])
    return full, partial

def filter_ckpt_to_windows(ckpt, window_nums):
    window_nums = set(window_nums)
    return {
        'results': [r for r in ckpt['results'] if r['window'] in window_nums],
        'cv_match_predictions': [m for m in ckpt['cv_match_predictions'] if m['window'] in window_nums],
    }

ckpt_ml = load_ckpt(WP011 / 'cv_checkpoint_multileague.pkl')
if ckpt_ml['results']:
    full_windows, partial_windows = leagues_coverage_split(ckpt_ml)
    print(f'multileague: {len(full_windows)} windows with every league present, '
          f'{len(partial_windows)} with at least one skipped (trimmed leagues not started yet)')

    for label, wins in [('all-leagues-present windows', full_windows), ('partial-coverage windows', partial_windows)]:
        if not wins:
            print(f'  {label}: none')
            continue
        sub_ckpt = filter_ckpt_to_windows(ckpt_ml, wins)
        r = gap_vs_pinnacle(sub_ckpt, windows)
        print(f"  {label:<28} n={r['n']:>3}  model RPS {r['model_rps']:.4f}  gap {r['gap']:+.4f}  "
              f"CI [{r['ci'][0]:+.4f}, {r['ci'][1]:+.4f}]")

    # same split applied to baseline/lineup_loose_combo's SAME window numbers,
    # so the full-coverage comparison is apples-to-apples against arms that
    # were never affected by trimming in the first place.
    for name, ckpt_name in [('baseline', 'baseline'), ('lineup_loose_combo', 'lineup_loose_combo')]:
        ckpt = load_ckpt(WP011 / f'cv_checkpoint_{ckpt_name}.pkl')
        if not ckpt['results'] or not full_windows:
            continue
        sub_ckpt = filter_ckpt_to_windows(ckpt, full_windows)
        r = gap_vs_pinnacle(sub_ckpt, windows)
        print(f"  {name} (same {len(full_windows)} windows) n={r['n']:>3}  model RPS {r['model_rps']:.4f}  "
              f"gap {r['gap']:+.4f}  CI [{r['ci'][0]:+.4f}, {r['ci'][1]:+.4f}]")
else:
    print('multileague arm not run yet')

multileague: 11 windows with every league present, 7 with at least one skipped (trimmed leagues not started yet)
  all-leagues-present windows  n=111  model RPS 0.1938  gap +0.0157  CI [+0.0065, +0.0254]
  partial-coverage windows     n= 84  model RPS 0.1895  gap +0.0100  CI [-0.0014, +0.0212]
  baseline (same 11 windows) n=111  model RPS 0.1932  gap +0.0151  CI [+0.0061, +0.0246]
  lineup_loose_combo (same 11 windows) n=111  model RPS 0.1921  gap +0.0140  CI [+0.0051, +0.0234]


## Phase 3 — full 35-window confirmation

Only worth running if Phase 2 screening shows the `multileague` arm is at least competitive with `lineup_loose_combo` — per this project's standing rule (WP006/WP009/WP010), a string of nulls at screening is a real stop signal, not a reason to spend the full compute anyway. **Heavy compute — run this yourself, and only after looking at Phase 2's numbers.**

In [10]:
FULL_WINDOWS = list(range(1, len(windows) + 1))
seed_from(WP001 / 'cv_checkpoint.pkl', 'full_baseline', FULL_WINDOWS)
seed_from(WP009 / 'cv_checkpoint_lineup_loose_combo.pkl', 'full_lineup_loose_combo', FULL_WINDOWS)

ckpt_full_multileague = WP011 / 'cv_checkpoint_full_multileague.pkl'
t0 = time.time()
run_windows_concurrent(
    SCRIPT, DATA_PATH, ckpt_full_multileague, FULL_WINDOWS,
    config_overrides=ARMS['multileague'], max_workers=MAX_WORKERS, timeout=WINDOW_TIMEOUT,
)
print(f'\nfull CV wall time: {(time.time()-t0)/60:.1f} min')

for name, ckpt_name in [('baseline', 'full_baseline'), ('lineup_loose_combo', 'full_lineup_loose_combo'), ('multileague', 'full_multileague')]:
    ckpt = load_ckpt(WP011 / f'cv_checkpoint_{ckpt_name}.pkl')
    if not ckpt['results']:
        print(f'{name}: not run yet')
        continue
    r = gap_vs_pinnacle(ckpt, windows)
    print(f"{name:<20} n={r['n']:>3}  model RPS {r['model_rps']:.4f}  gap {r['gap']:+.4f}  "
          f"CI [{r['ci'][0]:+.4f}, {r['ci'][1]:+.4f}]")

seeded full_baseline from cv_checkpoint.pkl: 35 windows
seeded full_lineup_loose_combo from cv_checkpoint_lineup_loose_combo.pkl: 18 windows


/Users/hadiahmed/Documents/projects/football-predictor/venv/lib/python3.13/site-packages/arviz/__init__.py:39: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(
/Users/hadiahmed/Documents/projects/football-predictor/venv/lib/python3.13/site-packages/arviz/__init__.py:39: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(
/Users/hadiahmed/Documents/projects/football-predictor/venv/lib/python3.13/site-packages/arviz/__init__.py:39: FutureWarning: 
ArviZ is undergoing a major refac

[window 3/35] multi-league (['Bundesliga', 'EPL', 'La_Liga', 'Ligue_1', 'Serie_A'], eval=EPL) training rounds 1-46 (use_xg=True, use_dc=True, overrides={})
[window 4/35] multi-league (['Bundesliga', 'EPL', 'La_Liga', 'Ligue_1', 'Serie_A'], eval=EPL) training rounds 1-51 (use_xg=True, use_dc=True, overrides={})
[window 2/35] multi-league (['Bundesliga', 'EPL', 'La_Liga', 'Ligue_1', 'Serie_A'], eval=EPL) training rounds 1-41 (use_xg=True, use_dc=True, overrides={})
[window 1/35] multi-league (['Bundesliga', 'EPL', 'La_Liga', 'Ligue_1', 'Serie_A'], eval=EPL) training rounds 1-36 (use_xg=True, use_dc=True, overrides={})
  prepare_multileague_data: skipping 'Bundesliga' for this window — no matches on/before 2021-08-29 (league's fetched history starts later)
  prepare_multileague_data: skipping 'La_Liga' for this window — no matches on/before 2021-08-29 (league's fetched history starts later)
  prepare_multileague_data: skipping 'Serie_A' for this window — no matches on/before 2021-08-29 (l

Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:03<?, ?it/s]

Running chain 2:   0%|          | 0/4000 [00:03<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:03<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:03<?, ?it/s]

Running chain 2:   0%|          | 0/4000 [00:03<?, ?it/s]

Running chain 0:   5%|▌         | 200/4000 [00:10<02:04, 30.62it/s]

Running chain 2:   5%|▌         | 200/4000 [00:10<02:11, 28.84it/s]

Running chain 0:   5%|▌         | 200/4000 [00:11<02:42, 23.35it/s]

Running chain 2:  10%|█         | 400/4000 [00:11<01:06, 53.97it/s]

Running chain 2:   5%|▌         | 200

[window 1] MAE=1.147 LL_improvement=1.12
[window 1] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp011_multileague_hierarchy/.tmp_cv_checkpoint_full_multileague_w1.pkl


...

[window 34] MAE=0.873 LL_improvement=0.76
[window 34] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp011_multileague_hierarchy/.tmp_cv_checkpoint_full_multileague_w34.pkl




Running chain 2:  65%|██████▌   | 2600/4000 [11:07<03:12,  7.29it/s]

  [window 34] done and merged



Running chain 0:  70%|███████   | 2800/4000 [11:26<02:38,  7.56it/s]

Running chain 0:  75%|███████▌  | 3000/4000 [11:49<02:07,  7.82it/s]

Running chain 0:  80%|████████  | 3200/4000 [12:13<01:39,  8.01it/s]

Running chain 0:  85%|████████▌ | 3400/4000 [12:36<01:13,  8.16it/s]

Running chain 0:  90%|█████████ | 3600/4000 [13:00<00:48,  8.27it/s]

Running chain 0:  95%|█████████▌| 3800/4000 [13:23<00:23,  8.35it/s]

Running chain 0: 100%|██████████| 4000/4000 [13:47<00:00,  4.84it/s]


Running chain 2: 100%|██████████| 4000/4000 [13:51<00:00,  4.81it/s]

Running chain 1: 100%|██████████| 4000/4000 [13:55<00:00,  4.79it/s]
We recommend running at least 4 chains for robust computation of convergence diagnostics


[window 35] MAE=0.900 LL_improvement=0.92
[window 35] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp011_multileague_hierarchy/.tmp_cv_checkpoint_full_multileague_w35.pkl
  [window 35] done and merged

full CV wall time: 72.1 min
baseline             n=361  model RPS 0.1925  gap +0.0139  CI [+0.0080, +0.0198]
lineup_loose_combo   n=195  model RPS 0.1903  gap +0.0117  CI [+0.0047, +0.0186]
multileague          n=361  model RPS 0.1926  gap +0.0141  CI [+0.0081, +0.0200]
